# Clinical & Manual Quality Review Dashboard: Chest X-Ray Synthetics

This notebook provides a manual verification interface for supervising chest X-ray synthetic variations generated by Stable Diffusion (Week 5). 
It allows clinicians and supervisors to:
- Inspect generated chest radiographs side-by-side with original conditioning real images.
- Track and plot Fréchet Inception Distance (FID) quality distributions.
- Check off qualitative metrics such as anatomical correctness, absence of visual artifacts, and clinical generalizability.

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

synthetic_csv = '../data/synthetic/synthetic_metadata.csv'
if os.path.exists(synthetic_csv):
    df = pd.read_csv(synthetic_csv)
    print(f"Loaded {len(df)} accepted synthetic Pneumothorax images metadata.")
    display(df.head())
else:
    print("synthetic_metadata.csv not found. Please execute synthetic generation script first.")

## 1. Visual Comparison Grid: Real vs Synthetic variations

In [ ]:
import os

import matplotlib.image as mpimg
import matplotlib.pyplot as plt

def _first_images(d, k):
    if not os.path.isdir(d):
        return []
    fs = [f for f in sorted(os.listdir(d)) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    return [os.path.join(d, f) for f in fs[:k]]

def plot_comparison_grid(real_dir, synth_dir, n_samples=3):
    reals = _first_images(real_dir, n_samples)
    synths = _first_images(synth_dir, n_samples)
    if not reals or not synths:
        raise FileNotFoundError(
            f'No real/synthetic images found in {real_dir} / {synth_dir}. '
            'Run scripts/generate_synthetics.py first (this cell never fabricates noise).'
        )
    rows = min(n_samples, len(reals), len(synths))
    fig, axes = plt.subplots(rows, 2, figsize=(10, rows * 4))
    axes = axes.reshape(rows, 2)
    for i in range(rows):
        axes[i, 0].imshow(mpimg.imread(reals[i]), cmap='gray')
        axes[i, 0].set_title(f'Real conditioning image {i + 1}')
        axes[i, 0].axis('off')
        axes[i, 1].imshow(mpimg.imread(synths[i]), cmap='gray')
        axes[i, 1].set_title(f'Synthesized SD variation {i + 1}')
        axes[i, 1].axis('off')
    plt.tight_layout()
    plt.show()

plot_comparison_grid('../data/raw/images', '../data/synthetic/accepted', n_samples=3)


## 2. Inception-based FID Score Distribution
Fréchet Inception Distance (FID) evaluates the distribution of synthesized images against genuine ones. Lower scores indicate superior anatomical realism and matching textures.

In [ ]:
import os

import matplotlib.pyplot as plt
import pandas as pd

# REAL FID values logged per batch by scripts/generate_synthetics.py.
_fid_log = next((p for p in ['../data/synthetic/fid_log.csv', 'data/synthetic/fid_log.csv']
                 if os.path.exists(p)), None)
if _fid_log is None:
    raise FileNotFoundError(
        'data/synthetic/fid_log.csv not found. Run scripts/generate_synthetics.py first '
        '(this cell never uses hardcoded FID numbers).'
    )
fid_df = pd.read_csv(_fid_log)
threshold = 50.0
labels = [f'Batch {i + 1}' for i in range(len(fid_df))]
scores = fid_df['fid'].tolist()
plt.figure(figsize=(8, 4.5))
plt.bar(labels, scores, color=['red' if f > threshold else 'dodgerblue' for f in scores], alpha=0.85)
plt.axhline(y=threshold, color='crimson', linestyle='--', linewidth=1.5, label='FID Quality Threshold (50.0)')
plt.title('Synthetic Batch Quality Gate (Inception FID) - real values', fontsize=12, fontweight='bold')
plt.ylabel('FID Score (lower is better)')
plt.legend()
plt.grid(axis='y', linestyle=':', alpha=0.6)
plt.show()


## 3. Clinician Verification Checklists

Please evaluate the generated batch based on the following professional radiology metrics:

- `[ ]` **Anatomical Correctness**: Are ribs, diaphragms, clavicles, and heart contours clearly defined without distorted, unnatural structures?
- `[ ]` **Pneumothorax Fidelity**: Are the lung outlines, collapsed margins, and pleural air collections physically consistent and visually detectable?
- `[ ]` **Absence of Text/Artifacts**: Are there no spurious letters, horizontal scanning streaks, or high-contrast boundary annotations created by the generator?
- `[ ]` **Contrast Realism**: Does the density and gray-level scale accurately replicate standard clinical DICOM settings?